# CODE 1.4: Check Splits

> **Goal:** make sure the split files are correct, and that no data can leak between train, validation and test, before any model uses them.
>
> - **Input:** `splits/split_1.csv`, `splits/split_2.csv`, `splits/split_3.csv` and `splits/same_gum_photos.csv`
> - **Output:** no files. Only a list of checks. At the end you should see **ALL CHECKS PASSED**.

## Import Libraries & Variables

### Import Libraries

> - `os`: works with folders and file paths.
> - `glob`: finds all files whose name matches a pattern, like `split_*.csv`.
> - `pandas`: reads the split files.

In [1]:
import os
import glob

import pandas as pd

### Variables

#### Folder Paths

> This code finds the main project folder. It starts in the folder of this notebook and goes up until it finds the `data` folder, so the paths work wherever the notebook is.
>
> - `ROOT_DIR`: the main project folder.
> - `RAW_DIR`: the original data. We only read from it.
> - `GUM_DIR`: the chewing gum photos, masks and photo list.
> - `SPLITS_DIR`: where our split files are saved.
> - `SAME_GUM_FILE`: the list of photos that show the same gum (notebook 1.2).
>
> If the data is missing, the code stops with a message that says what to do.

In [2]:
ROOT_DIR = os.getcwd()
while not os.path.isdir(os.path.join(ROOT_DIR, "data")) and ROOT_DIR != os.path.dirname(ROOT_DIR):
    ROOT_DIR = os.path.dirname(ROOT_DIR)

RAW_DIR = os.path.join(ROOT_DIR, "data", "raw")
GUM_DIR = os.path.join(RAW_DIR, "chewinggum")
SPLITS_DIR = os.path.join(ROOT_DIR, "splits")
SAME_GUM_FILE = os.path.join(SPLITS_DIR, "same_gum_photos.csv")

if not os.path.exists(os.path.join(GUM_DIR, "image_anno.csv")):
    raise FileNotFoundError("Data not found. Copy the 'chewinggum' folder into data/raw/ (see README).")
print("Project folder found:", os.path.basename(ROOT_DIR))

Project folder found: visual-defect-inspection


#### Expected Values

> The numbers every split file must have, all taken from notebook 1.3: the test and validation sizes are its Split Settings, the 6 excluded photos are printed in "Load the Photos of the Same Gum", and the train counts are printed in "Make 3 Splits (Preview)".
>
> - `PARTS_IN_USE`: the parts a model may use. `excluded` photos are never used.

In [3]:
N_PHOTOS = 603
EXPECTED = {
    "train": (347, 40),
    "val": (50, 10),
    "test": (100, 50),
    "excluded": (6, 0),
}
PARTS_IN_USE = ["train", "val", "test"]
LABEL_STEPS = [5, 10, 20, 40]
N_TYPES = 5

split_files = sorted(glob.glob(os.path.join(SPLITS_DIR, "split_*.csv")))
print("Split files found:", [os.path.basename(f) for f in split_files])

Split files found: ['split_1.csv', 'split_2.csv', 'split_3.csv']


## Functions

### Record a Check

> `check()` prints PASS or FAIL for one check and stores the result in `results`, with the name of the check as the key. Every name starts with the split file, so the same check on two files is stored twice, but running a cell again never counts a check twice.

In [4]:
results = {}

def check(name, ok):
    results[name] = bool(ok)
    print(("PASS  " if ok else "FAIL  ") + name)

### Read a Split File

> `read_split()` reads one split file. Empty cells become empty text, and `label_order` is read as whole numbers.

In [5]:
def read_split(path):
    df = pd.read_csv(path, dtype={"label_order": "Int64"})
    return df.fillna({"mask": "", "defect_types": "", "same_gum_as": ""})

### Count Photos in a Part

> `count_part()` counts the normal (`is_defect` = `False`) or defect (`is_defect` = `True`) photos in one part of a split.

In [6]:
def count_part(df, part, is_defect):
    return int(((df["split"] == part) & (df["is_defect"] == is_defect)).sum())

## Run the Checks

### Check Every Split File

> First we check that there are 3 split files. Then, for each file:
>
> - **Counts:** the right number of photos, each photo once, and the right number of normal and defect photos in each part.
> - **Files:** every photo and mask still exists.
> - **Defect types:** all 5 types are in train, in validation and in test.
> - **YOLO sets:** `yolo_k` has exactly k defect photos + all normal train photos, and each set includes the smaller one.

In [7]:
check("3 split files found (one per seed)", len(split_files) == 3)

for path in split_files:
    df = read_split(path)
    name = os.path.splitext(os.path.basename(path))[0]
    print(f"\n=== {name} ===")

    # Counts
    check(f"{name}: {N_PHOTOS} photos", len(df) == N_PHOTOS)
    check(f"{name}: each photo appears only once", df["image"].is_unique)
    check(f"{name}: each photo is in train, val, test or excluded", df["split"].isin(list(EXPECTED)).all())
    for part, (n_normal, n_defect) in EXPECTED.items():
        check(f"{name}: {part} has {n_normal} normal + {n_defect} defect photos",
              count_part(df, part=part, is_defect=False) == n_normal
              and count_part(df, part=part, is_defect=True) == n_defect)

    # Files
    check(f"{name}: every photo file exists", all(os.path.exists(os.path.join(RAW_DIR, p)) for p in df["image"]))
    check(f"{name}: every mask file exists",
          all(os.path.exists(os.path.join(RAW_DIR, p)) for p in df.loc[df["is_defect"], "mask"]))

    # Defect types
    for part in PARTS_IN_USE:
        types = set(df.loc[(df["split"] == part) & df["is_defect"], "defect_types"].str.split(",").explode())
        check(f"{name}: all {N_TYPES} defect types are in {part}", len(types) == N_TYPES)

    # YOLO sets
    n_normal_train = EXPECTED["train"][0]
    previous = set()
    for k in LABEL_STEPS:
        used = df[df[f"yolo_{k}"]]
        check(f"{name}: yolo_{k} has {k} defect + {n_normal_train} normal photos, all from train",
              used["is_defect"].sum() == k and (~used["is_defect"]).sum() == n_normal_train
              and (used["split"] == "train").all())
        current = set(used["image"])
        check(f"{name}: yolo_{k} includes every photo of the smaller set", previous <= current)
        previous = current

    first_five = df[df["yolo_5"] & df["is_defect"]]
    check(f"{name}: the first 5 defect photos come from 5 different type groups", first_five["type_group"].nunique() == 5)

PASS  3 split files found (one per seed)

=== split_1 ===
PASS  split_1: 603 photos
PASS  split_1: each photo appears only once
PASS  split_1: each photo is in train, val, test or excluded
PASS  split_1: train has 347 normal + 40 defect photos
PASS  split_1: val has 50 normal + 10 defect photos
PASS  split_1: test has 100 normal + 50 defect photos
PASS  split_1: excluded has 6 normal + 0 defect photos
PASS  split_1: every photo file exists
PASS  split_1: every mask file exists
PASS  split_1: all 5 defect types are in train
PASS  split_1: all 5 defect types are in val
PASS  split_1: all 5 defect types are in test
PASS  split_1: yolo_5 has 5 defect + 347 normal photos, all from train
PASS  split_1: yolo_5 includes every photo of the smaller set
PASS  split_1: yolo_10 has 10 defect + 347 normal photos, all from train
PASS  split_1: yolo_10 includes every photo of the smaller set
PASS  split_1: yolo_20 has 20 defect + 347 normal photos, all from train
PASS  split_1: yolo_20 includes every 

### Data Leakage Checks

> **Data leakage** means that information from the validation or test photos reaches a model while it learns or while we choose its settings. The score then looks better than it really is. These checks make sure the split files cannot cause that:
>
> 1. **No photo is in two parts.** Already checked above: every photo appears only once, with one part.
> 2. **The left-out photos are never used:** every photo that notebook 1.2 left out is `excluded` in every split file, and no YOLO set uses it.
> 3. **No gum is in two parts:** for every group of photos of the same gum (notebook 1.2), all photos that are used sit in the same part.
> 4. **YOLO only learns from train photos:** no validation or test photo is in a YOLO set.
> 5. **Both models learn from the same normal photos:** the normal photos in every YOLO set are exactly PatchCore's photos (normal + train). This keeps the comparison fair.
> 6. **All 3 split files leave out the same photos.**

In [8]:
same_gum = pd.read_csv(SAME_GUM_FILE).fillna({"same_gum_as": ""})
left_out = set(same_gum.loc[~same_gum["keep"], "image"])
yolo_columns = [f"yolo_{k}" for k in LABEL_STEPS]
print(f"Photos of the same gum: {len(same_gum)} in {same_gum['group'].nunique()} groups | left out: {len(left_out)}")

excluded_per_file = {}
for path in split_files:
    df = read_split(path)
    name = os.path.splitext(os.path.basename(path))[0]
    print(f"\n=== {name} ===")
    part_of = dict(zip(df["image"], df["split"]))

    excluded = set(df.loc[df["split"] == "excluded", "image"])
    excluded_per_file[name] = excluded
    check(f"{name}: every photo left out in notebook 1.2 is excluded", excluded == left_out)
    check(f"{name}: excluded photos are never in a YOLO set",
          not df.loc[df["split"] == "excluded", yolo_columns].to_numpy().any())

    gums_in_two_parts = 0
    for _, group in same_gum.groupby("group"):
        used_parts = {part_of[p] for p in group["image"] if part_of[p] != "excluded"}
        gums_in_two_parts += len(used_parts) > 1
    check(f"{name}: no gum is in two parts", gums_in_two_parts == 0)

    for part in ["val", "test"]:
        check(f"{name}: {part} photos are never in a YOLO set",
              not df.loc[df["split"] == part, yolo_columns].to_numpy().any())

    patchcore_photos = set(df.loc[(df["split"] == "train") & ~df["is_defect"], "image"])
    same_normal = all(set(df.loc[df[c] & ~df["is_defect"], "image"]) == patchcore_photos for c in yolo_columns)
    check(f"{name}: PatchCore and every YOLO set learn from the same normal photos", same_normal)

print()
check("all 3 split files exclude the same photos", len({frozenset(v) for v in excluded_per_file.values()}) == 1)

Photos of the same gum: 12 in 6 groups | left out: 6

=== split_1 ===
PASS  split_1: every photo left out in notebook 1.2 is excluded
PASS  split_1: excluded photos are never in a YOLO set
PASS  split_1: no gum is in two parts
PASS  split_1: val photos are never in a YOLO set
PASS  split_1: test photos are never in a YOLO set
PASS  split_1: PatchCore and every YOLO set learn from the same normal photos

=== split_2 ===
PASS  split_2: every photo left out in notebook 1.2 is excluded
PASS  split_2: excluded photos are never in a YOLO set
PASS  split_2: no gum is in two parts
PASS  split_2: val photos are never in a YOLO set
PASS  split_2: test photos are never in a YOLO set
PASS  split_2: PatchCore and every YOLO set learn from the same normal photos

=== split_3 ===
PASS  split_3: every photo left out in notebook 1.2 is excluded
PASS  split_3: excluded photos are never in a YOLO set
PASS  split_3: no gum is in two parts
PASS  split_3: val photos are never in a YOLO set
PASS  split_3: te

## More Information

### Defect Types in Each Part

> Not a check, just information: how many photos of each defect type are in each part. A photo with two types is counted once for each type.

In [9]:
for path in split_files:
    df = read_split(path)
    defects = df[df["is_defect"]].copy()
    defects["type"] = defects["defect_types"].str.split(",")
    defects = defects.explode("type").reset_index(drop=True)
    print(f"=== {os.path.splitext(os.path.basename(path))[0]} ===")
    print(pd.crosstab(defects["type"], defects["split"])[PARTS_IN_USE].to_string(), "\n")

=== split_1 ===
split                 train  val  test
type                                  
chunk of gum missing     21    4    22
corner missing           15    4    17
scratches                 6    2     8
similar colour spot      10    3    12
small cracks             12    3    14 

=== split_2 ===
split                 train  val  test
type                                  
chunk of gum missing     19    5    23
corner missing           13    4    19
scratches                 6    2     8
similar colour spot       9    2    14
small cracks             13    2    14 

=== split_3 ===
split                 train  val  test
type                                  
chunk of gum missing     22    4    21
corner missing           14    4    18
scratches                 7    1     8
similar colour spot       9    2    14
small cracks             11    4    14 



### How Different Are the Test Sets?

> The three splits have different test photos on purpose. This counts how many test photos two splits share.

In [10]:
n_test = sum(EXPECTED["test"])
test_photos = {os.path.splitext(os.path.basename(f))[0]: set(read_split(f).query("split == 'test'")["image"])
               for f in split_files}
names = list(test_photos)

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        shared = len(test_photos[names[i]] & test_photos[names[j]])
        print(f"{names[i]} and {names[j]}: {shared} of {n_test} test photos are the same")

split_1 and split_2: 51 of 150 test photos are the same
split_1 and split_3: 48 of 150 test photos are the same
split_2 and split_3: 49 of 150 test photos are the same


## Result

### Final Result

> If every check passed, the splits are ready to use. If a check failed, its name is printed.

In [11]:
failed = [name for name, ok in results.items() if not ok]
if results and not failed:
    print(f"ALL CHECKS PASSED ({len(results)} checks)")
else:
    print(f"{len(failed)} of {len(results)} checks FAILED:")
    for name in failed:
        print("  -", name)

ALL CHECKS PASSED (83 checks)


## Summary

> - All 3 split files have the right sizes, and every file they point to exists.
> - All 5 defect types are in train, in validation and in test in every split.
> - The YOLO sets grow step by step (5, 10, 20, 40).
> - No data can leak through the split files: no photo and no gum is in two parts, the 6 left-out photos are never used, and YOLO never learns from validation or test photos.
> - PatchCore and YOLO learn from the same 347 normal photos.
> - The three test sets are different: two splits share between 48 and 51 of their 150 test photos.
>
> **Next:** `notebooks/2_eda/2.1_explore_data.ipynb`